# XGBoost -- train

In [ ]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, balanced_accuracy_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

NUM_CLASSES = 5

def train_xgboost(X_flat_train, y_flat_train,
                  X_flat_val=None, y_flat_val=None,
                  n_estimators=600, max_depth=6,
                  learning_rate=0.05, subsample=0.8,
                  colsample_bytree=0.8):
    model = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=5,
        max_delta_step=2,  
        subsample=0.7,
        colsample_bytree=0.7,
        colsample_bylevel=0.7,
        reg_alpha=1.0,     # L1
        reg_lambda=3.0,    # L2
        gamma=0.1,
        device            = DEV if torch.cuda.is_available() else "cpu",
        objective='multi:softprob',
        eval_metric       = ["mlogloss", "merror"],
        num_class = 5,
        tree_method='hist',
        max_bin=256,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=200
    )

    eval_set = []
    if X_flat_val is not None:
        eval_set = [(X_flat_val, y_flat_val)]

    model.fit(
        X_flat_train, y_flat_train,
        eval_set   = eval_set if eval_set else [(X_flat_train, y_flat_train)],
        verbose    = 100,
    )
    log.info(f"XGBoost best iteration: {model.best_iteration}")
    return model




def compute_class_weights(y: np.ndarray, n_classes: int) -> torch.Tensor:
    counts = np.bincount(y, minlength=n_classes).astype(float)
    weights = counts.sum() / (n_classes * counts + 1e-6)
    return torch.tensor(weights, dtype=torch.float32)



In [ ]:
xgb_model = train_xgboost(X_tr_aug[:,-1,:], y_flat_tr,   X_val_aug[:,-1,:], y_flat_val)
